In [56]:
import sympy as sym
from sympy import *
from scipy import linalg as Sc
import numpy as np
from tabulate import tabulate

In [57]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [58]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g


In [59]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    eigenvaluesx, eigenvectorsx = Sc.eig(Tensor)
    print(' Unsorted Eigenvalues from scipy:\n', eigenvaluesx, '\n')
    print(' Unsorted Eigenvectors from scipy:\n', eigenvectorsx, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [60]:
#constants for 11B nuclei
Ispin = float(3/2)
w0 = 192.55 #Larmor Frequency for 11B (MHz)
wkhz = w0*10**3 # LArmor frequency in Hz

#factor q given in article
q = (3-4*Ispin*(Ispin + 1))/(16*(wkhz))

#Coefficient for LHQ (cluster1) from ASICS (in kHz)
A_coeff = [-2.237215, -2.630916,-3.326333]
B_coeff = [1.436043, 2.462382,-1.178245]
C_coeff = [-3.527478, 0.271774,-0.704537]
D_coeff = [0.222557, -0.401864,-0.925624]
E_coeff = [0.004629, -0.525906,-1.128563]

# A_coeff = [-2.237215, -2.744504,-3.396561]
# B_coeff = [1.436043, 2.561384,-1.133846]
# C_coeff = [-3.527478, 0.986618,-0.473004]
# D_coeff = [0.222557, -0.376258,-0.980912]
# E_coeff = [0.004629, -0.579924,-1.166911]

# A_coeff = [-2.237215, -2.630916,-3.396561]
# B_coeff = [1.436043, 2.462382,-1.133846]
# C_coeff = [-3.527478, 0.271774,-0.473004]
# D_coeff = [0.222557, -0.401864,-0.980912]
# E_coeff = [0.004629, -0.525906,-1.166911]




In [61]:
#Define symbol for quadrupolar tensor and force them to be real
AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q = sym.symbols('AzzmAyy_Q,AzzmAxx_Q,AyymAxx_Q,Ayz_Q,Axz_Q,Axy_Q', real=True)

#Variable for each equation set
quad_tensor = [(AzzmAyy_Q, Ayz_Q), (AzzmAxx_Q, Axz_Q), (AyymAxx_Q, Axy_Q)]

# List to hold solutions for quadrupolar tensor terms
solutions_Q = []

for i, (diagonal_diff, off_diagonal) in enumerate(quad_tensor):
    
    eq1 = sym.Eq(-((diagonal_diff)**2 - 4*(off_diagonal)**2)*((9*q/8)), D_coeff[i])
    eq2 = sym.Eq(off_diagonal*(diagonal_diff)*(9*q/2), E_coeff[i])

    # Solve the system
    solution = sym.solve([eq1, eq2], (diagonal_diff, off_diagonal))
    solutions_Q.append(solution)

# Assign the solutions to the respective variables
AzzmAyy_Q = [solutions_Q[0][0][0], solutions_Q[0][1][0]]
Ayz_Q = [solutions_Q[0][0][1], solutions_Q[0][1][1]]
AzzmAxx_Q = [solutions_Q[1][0][0], solutions_Q[1][1][0]]
Axz_Q = [solutions_Q[1][0][1], solutions_Q[1][1][1]]
AyymAxx_Q = [solutions_Q[2][0][0], solutions_Q[2][1][0]]
Axy_Q = [solutions_Q[2][0][1], solutions_Q[2][1][1]]

# print(solutions_Q)
# Print the final results for the variables
print(f"Azz - Ayy: {AzzmAyy_Q}, Ayz: {Ayz_Q}")
print(f"Azz - Axx: {AzzmAxx_Q}, Axz: {Axz_Q}")
print(f"Ayy - Axx: {AyymAxx_Q}, Axy: {Axy_Q}")  



Azz - Ayy: [-225.376680894578, 225.376680894578], Ayz: [1.17178503638428, -1.17178503638428]
Azz - Axx: [-172.242964530424, 172.242964530424], Axz: [-174.195278639089, 174.195278639089]
Ayy - Axx: [-246.837437131111, 246.837437131111], Axy: [-260.846206433757, 260.846206433757]


In [62]:
import itertools
# Generate all combinations of AzzmAxx, AyymAxx, and AzzmAyy
combinations = list(itertools.product(AzzmAxx_Q, AyymAxx_Q, AzzmAyy_Q))
print('Combination of (Azz - Axx), (Ayy - Axx), (Azz - Ayy): \n ', combinations)

Combination of (Azz - Axx), (Ayy - Axx), (Azz - Ayy): 
  [(-172.242964530424, -246.837437131111, -225.376680894578), (-172.242964530424, -246.837437131111, 225.376680894578), (-172.242964530424, 246.837437131111, -225.376680894578), (-172.242964530424, 246.837437131111, 225.376680894578), (172.242964530424, -246.837437131111, -225.376680894578), (172.242964530424, -246.837437131111, 225.376680894578), (172.242964530424, 246.837437131111, -225.376680894578), (172.242964530424, 246.837437131111, 225.376680894578)]


In [63]:

#Find Quadrupolar tensor diagonal elements

Axx1 = []; Axx2 = []; Axx3 = []
Ayy1 = []; Ayy2 = []; Ayy3 = []
Azz1 = []; Azz2 = []; Azz3 = []

# Initialize variables to track the best combination and minimum variation
best_combination = None
min_variation = float('inf')
for (AzzmAxx_val, AyymAxx_val, AzzmAyy_val) in combinations:
    # Solution 1
    Axx1_val = (-(AzzmAxx_val + AyymAxx_val)/3)
    Ayy1_val = Axx1_val + AyymAxx_val
    Azz1_val = Axx1_val + AzzmAxx_val

    #Save values
    Axx1.append(Axx1_val)
    Ayy1.append(Ayy1_val)
    Azz1.append(Azz1_val)

     # Solution 2
    Ayy2_val = -(AzzmAyy_val - AyymAxx_val) / 3
    Axx2_val = Ayy2_val - AyymAxx_val
    Azz2_val = Ayy2_val + AzzmAyy_val

     #Save values
    Axx2.append(Axx2_val)
    Ayy2.append(Ayy2_val)
    Azz2.append(Azz2_val)

    # Solution 3
    Azz3_val = (AzzmAxx_val + AzzmAyy_val) / 3
    Axx3_val = Azz3_val - AzzmAxx_val
    Ayy3_val = Azz3_val - AzzmAyy_val
    
    #Save values
    Axx3.append(Axx3_val)
    Ayy3.append(Ayy3_val)
    Azz3.append(Azz3_val)

    # Convert sympy Float to regular Python float for NumPy functions
    Axx1_val = float(Axx1_val)
    Axx2_val = float(Axx2_val)
    Axx3_val = float(Axx3_val)
    
    Ayy1_val = float(Ayy1_val)
    Ayy2_val = float(Ayy2_val)
    Ayy3_val = float(Ayy3_val)
    
    Azz1_val = float(Azz1_val)
    Azz2_val = float(Azz2_val)
    Azz3_val = float(Azz3_val)

    # Calculate variation (standard deviation) for Axx, Ayy, Azz
    variation_Axx = np.std([Axx1_val, Axx2_val, Axx3_val])
    variation_Ayy = np.std([Ayy1_val, Ayy2_val, Ayy3_val])
    variation_Azz = np.std([Azz1_val, Azz2_val, Azz3_val])

    total_variation = variation_Axx + variation_Ayy + variation_Azz

    # Update the best combination if the current one has less variation
    if total_variation < min_variation:
        min_variation = total_variation
        best_combination = (AzzmAxx_val, AyymAxx_val, AzzmAyy_val)
        best_Axx_Q = np.mean([Axx1_val, Axx2_val, Axx3_val])
        best_Ayy_Q = np.mean([Ayy1_val, Ayy2_val, Ayy3_val])
        best_Azz_Q = np.mean([Azz1_val, Azz2_val, Azz3_val])

#Get index for off-diagonal elements        
index_AzzmAxx = AzzmAxx_Q.index(best_combination[0])
best_Axz_Q = Axz_Q[index_AzzmAxx]

index_AyymAxx = AyymAxx_Q.index(best_combination[1])
best_Axy_Q = Axy_Q[index_AyymAxx]

index_AzzmAyy = AzzmAyy_Q.index(best_combination[2])
best_Ayz_Q = Ayz_Q[index_AzzmAyy]

# Print results
print("Axx1:", Axx1)
print("Axx2:", Axx2)
print("Axx3:", Axx3)
print("================================")

print("Ayy1:", Ayy1)
print("Ayy2:", Ayy2)
print("Ayy3:", Ayy3)
print("================================")

print("Azz1:", Azz1)
print("Azz2:", Azz2)
print("Azz3:", Azz3)
print("================================")

print("Best combination with minimum standard deviation:")
print("AzzmAxx:", best_combination[0])
print("AyymAxx:", best_combination[1])
print("AzzmAyy:", best_combination[2])


print("Axz:", best_Axz_Q)
print("Axy:", best_Axy_Q)
print("Ayz:", best_Ayz_Q)

print("Average of Axx1, Axx2, Axx3 with minimum standard deviation:", best_Axx_Q)
print("Average of Ayy1, Ayy2, Ayy3 with minimum standard deviation:", best_Ayy_Q)
print("Average of Azz1, Azz2, Azz3 with minimum standard deviation:", best_Azz_Q)




Axx1: [139.693467220512, 139.693467220512, -24.8648242002290, -24.8648242002290, 24.8648242002290, 24.8648242002290, -139.693467220512, -139.693467220512]
Axx2: [239.683851718933, 89.4327311225483, -89.4327311225483, -239.683851718933, 239.683851718933, 89.4327311225483, -89.4327311225483, -239.683851718933]
Axx3: [39.7030827220903, 189.954203318475, 39.7030827220903, 189.954203318475, -189.954203318475, -39.7030827220903, -189.954203318475, -39.7030827220903]
Ayy1: [-107.143969910599, -107.143969910599, 221.972612930882, 221.972612930882, -221.972612930882, -221.972612930882, 107.143969910599, 107.143969910599]
Ayy2: [-7.15358541217786, -157.404706008563, 157.404706008563, 7.15358541217786, -7.15358541217786, -157.404706008563, 157.404706008563, 7.15358541217786]
Ayy3: [92.8367990862437, -207.665442106527, 92.8367990862437, -207.665442106527, 207.665442106527, -92.8367990862437, 207.665442106527, -92.8367990862437]
Azz1: [-32.5494973099125, -32.5494973099125, -197.107788730653, -197.1

In [64]:
#Define symbol for CSA tensor and force them to be real
Azz_s, Axx_s, Ayy_s, Ayz_s, Axz_s, Axy_s = sym.symbols('Azz_s,Axx_s,Ayy_s,Ayz_s,Axz_s,Axy_s', real=True)

#Variables for each equation
cs_tensor = [(Ayy_s, Azz_s, Ayz_s), # for x -> Abb = Ayy; Agg = Azz; Abg = Ayz
             (Axx_s, Azz_s, Axz_s), # for y -> Abb = Axx; Agg = Azz; Abg = Axz
             (Axx_s, Ayy_s, Axy_s)] # for z -> Abb = Axx; Agg = Ayy; Abg = Axy

#Store variables in dictionary for access
A = {
    'xx': best_Axx_Q, 'yy': best_Ayy_Q, 'zz': best_Azz_Q,
    'yz': best_Ayz_Q, 'zy': best_Ayz_Q,
    'xz': best_Axz_Q, 'zx': best_Axz_Q,
    'xy': best_Axy_Q, 'yx': best_Axy_Q,
}

#Define rotation tuple (a, b, g, bg, m)
rotations = [
    ('x', 'y', 'z', 'yz', 1),   # a = x, b = y, g = z, m = 1
    ('y', 'x', 'z', 'xz', 1),  # a = y, b = x, g = z, m = 1
    ('z', 'x', 'y', 'xy', -1)  # a = z, b = x, g = y, m = -1
]

# List to hold solutions
solutions_cs = []

for i, (Abb_s, Agg_s, Abg_s) in enumerate(cs_tensor):
    a, b, g, bg, m = rotations[i]
    eq1 = sym.Eq(
        (8*A[a+a]*(A[b+b] + A[g+g] - A[a+a]) + 16*(A[a+b]**2 + A[a+g]**2) + 5*(A[b+b]**2 + A[g+g]**2) + 28*A[b+g]**2 - 18*A[b+b]*A[g+g])*(q/8) - 0.5*(Abb_s + Agg_s)*wkhz, A_coeff[i]
        )
    
    eq2 = sym.Eq(
        m*(2*A[a+a]*(A[b+b] - A[g+g]) - 12*(A[a+b]**2 - A[a+g]**2) - A[b+b]**2 + A[g+g]**2)*(q/2) - 0.5*m*(Agg_s - Abb_s)*wkhz, B_coeff[i]
        )
    
    eq3 = sym.Eq(
        -m*(-2*A[a+a]*A[b+g] + 12*A[a+b]*A[a+g] + A[b+g]*(A[b+b] + A[g+g]))*q - m*Abg_s*wkhz, C_coeff[i]
    )
    # Solve the system
    solution = sym.solve([eq1, eq2, eq3], (Abb_s, Agg_s, Abg_s))
    solutions_cs.append(solution)

print(solutions_cs)

#saving solutions
Axx_s = np.mean([solutions_cs[1][Axx_s], solutions_cs[2][Axx_s]])
Ayy_s = np.mean([solutions_cs[0][Ayy_s], solutions_cs[2][Ayy_s]])
Azz_s = np.mean([solutions_cs[0][Azz_s], solutions_cs[1][Azz_s]])

Ayz_s = solutions_cs[0][Ayz_s]
Axz_s = solutions_cs[1][Axz_s]
Axy_s = solutions_cs[2][Axy_s]

print('Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s: \n', Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s)


[{Ayy_s: 1.01240812959789e-5, Azz_s: 5.84339274317027e-6, Ayz_s: 2.93597582800902e-5}, {Axx_s: 1.35740757818137e-5, Azz_s: 5.67916788641641e-6, Axz_s: -3.00123470927002e-6}, {Axx_s: 1.22780084127049e-5, Ayy_s: 7.08595474137057e-6, Axy_s: -3.32906420624373e-6}]
Axx_s, Ayy_s, Azz_s, Ayz_s, Axz_s, Axy_s: 
 1.29260420972593e-5 8.60501801867476e-6 5.76128031479334e-6 2.93597582800902e-5 -3.00123470927002e-6 -3.32906420624373e-6


In [65]:
#Quadrupolar Tensor in Tenon Frame
Q_T = np.zeros((3,3))
Q_T[0,0] = best_Axx_Q; Q_T[0,1] = best_Axy_Q; Q_T[0,2] = best_Axz_Q;
Q_T[1,0] = best_Axy_Q; Q_T[1,1] = best_Ayy_Q; Q_T[1,2] = best_Ayz_Q;
Q_T[2,0] = best_Axz_Q; Q_T[2,1] = best_Ayz_Q; Q_T[2,2] = best_Azz_Q;

#CSA tensor in tenon frame
CS_T = np.zeros((3,3))
CS_T[0,0] = Axx_s; CS_T[0,1] = Axy_s; CS_T[0,2] = Axz_s;
CS_T[1,0] = Axy_s; CS_T[1,1] = Ayy_s; CS_T[1,2] = Ayz_s;
CS_T[2,0] = Axz_s; CS_T[2,1] = Ayz_s; CS_T[2,2] = Azz_s;

print('Chemical Shift tensor (tenon frame): \n', CS_T*10**6,'\n')
print('Quadrupolar tensor (MHz) (tenon frame): \n', (Q_T*(2*(Ispin)*(2*(Ispin) - 1)))/10**3, '\n')

Chemical Shift tensor (tenon frame): 
 [[12.9260421  -3.32906421 -3.00123471]
 [-3.32906421  8.60501802 29.35975828]
 [-3.00123471 29.35975828  5.76128031]] 

Quadrupolar tensor (MHz) (tenon frame): 
 [[ 0.8381608  -1.56507724 -1.04517167]
 [-1.56507724 -0.94442824 -0.00703071]
 [-1.04517167 -0.00703071  0.10626743]] 



In [66]:
#following the Voseggard et al. paper for principal frame parameters JOURNAL OF MAGNETIC RESONANCE, Series A 122, 111 – 119 ( 1996 ) ARTICLE NO. 0186

#Calculate Quadrupolar Tensor in PAS

sorted_eigenvalues_Q, dc_Q, quad_avg, eigenvalues_Q, eigenvectors_Q = sort_eigenvalues(Q_T*(2*Ispin*(2*Ispin - 1))/10**3)
# print('Sorted Eigenvalues of Quadrupolar diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):\n', sorted_eigenvalues_Q, '\n') 

# Vyy = (sorted_eigenvalues_Q[0])*(2*Ispin*(2*Ispin - 1)) 
# Vxx = (sorted_eigenvalues_Q[1])*(2*Ispin*(2*Ispin - 1))
# Vzz = (sorted_eigenvalues_Q[2])*(2*Ispin*(2*Ispin - 1)) 

Vyy = (sorted_eigenvalues_Q[0])
Vxx = (sorted_eigenvalues_Q[1])
Vzz = (sorted_eigenvalues_Q[2]) 

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================ \n')

#Calculate CSA Tensor in PAS

sorted_eigenvalues_csa, dc_csa, csa_avg, eigenvalues_csa, eigenvectors_csa = sort_eigenvalues(CS_T*10**6)
# print('Sorted Eigenvalues of CSA diagonal matrix (|Ayy - Tr(A)/3| <= |Axx - Tr(A)/3| <= |Azz - Tr(A)/3|):\n', sorted_eigenvalues_csa, '\n') 

csyy = -(sorted_eigenvalues_csa[0]) 
csxx = -(sorted_eigenvalues_csa[1])
cszz = -(sorted_eigenvalues_csa[2]) 

print('CSA Tensor Components δyy, δxx, δzz: \n', csyy, csxx, cszz)
print('CSA direction cosine:\n', dc_csa)



 Unsorted Eigenvalues:
 [ 2.15701397 -2.00330078 -0.15371318] 

 Unsorted Eigenvectors:
 [[-0.8133898   0.53878225  0.21934155]
 [ 0.40952341  0.7981426  -0.44188117]
 [ 0.41314357  0.26959614  0.86984499]] 

 Unsorted Eigenvalues from scipy:
 [ 2.15701397+0.j -2.00330078+0.j -0.15371318+0.j] 

 Unsorted Eigenvectors from scipy:
 [[-0.8133898   0.53878225  0.21934155]
 [ 0.40952341  0.7981426  -0.44188117]
 [ 0.41314357  0.26959614  0.86984499]] 

Sorted Eigenvalues: 
 [-0.15371318 -2.00330078  2.15701397] 

Sorted Eigenvectors: 
 [[ 0.21934155  0.53878225 -0.8133898 ]
 [-0.44188117  0.7981426   0.40952341]
 [ 0.86984499  0.26959614  0.41314357]] 

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -0.15371318257239933 -2.0033007841735038 2.1570139667459025

 Unsorted Eigenvalues:
 [ 37.3976574   12.10613992 -22.21145688] 

 Unsorted Eigenvectors:
 [[ 0.18009761 -0.98364235 -0.00354775]
 [-0.71242587 -0.12795128 -0.68998394]
 [-0.67824349 -0.12679197  0.72381598]] 

 Unsorted Eigenvalues fr

In [67]:
#Quadrupolar tensor parameters
cq = Vzz
etaq = (Vyy - Vxx)/Vzz

#CSA tensor parameters
print(cszz, csyy, csxx)
iso_cs = np.mean([cszz, csyy, csxx]) 
csa = cszz - iso_cs

etas = (csyy - csxx)/csa


table = [['cq (MHz)', cq], ['etaq', etaq ], ['iso_cs (ppm)', iso_cs], ['csa (ppm)', csa], ['etas', etas] ] #converting Hz to ppm (Should be multiplied by 10**6)
print(tabulate(table, headers=['Quantity', 'Fit Value']))

22.211456884166694 -12.10613991938193 -37.397657395512184
Quantity        Fit Value
------------  -----------
cq (MHz)         2.15701
etaq             0.857476
iso_cs (ppm)    -9.09745
csa (ppm)       31.3089
etas             0.807806


In [68]:
print('\nEFG Direction Cosine\n')
print(dc_Q)

print('\nCS direction cosine\n')
print(dc_csa)

a_Q, b_Q, g_Q = get_euler_angles(dc_Q)
print("\nCalculated Euler angles (degrees):\n")
print('alpha:', a_Q, 'beta:', b_Q, 'gamma:', g_Q,'\n')


EFG Direction Cosine

[[ 0.53878225  0.21934155 -0.8133898 ]
 [ 0.7981426  -0.44188117  0.40952341]
 [ 0.26959614  0.86984499  0.41314357]]

CS direction cosine

[[ 0.18009761 -0.98364235 -0.00354775]
 [-0.71242587 -0.12795128 -0.68998394]
 [-0.67824349 -0.12679197  0.72381598]]

Calculated Euler angles (degrees):

alpha: 72.77991743675116 beta: 65.59753759142879 gamma: 26.724222917655762 



In [69]:
# Eigenvectors from ASICS values
alpha_efg = 290.8
beta_efg = 69
gamma_efg = 30

eigenvectors_efg = Rabc(alpha_efg, beta_efg, gamma_efg)
print('Eigenvectors for EFG tensor from ASICS\n',eigenvectors_efg)

get_euler_angles(eigenvectors_efg)

Eigenvectors for EFG tensor from ASICS
 [[ 0.57762233 -0.11257504 -0.80850437]
 [ 0.74595331  0.47503743  0.46679021]
 [ 0.33152091 -0.87273495  0.35836795]]


(-69.19999999999999, 69.0, 29.999999999999996)

In [70]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_Q), (dc_csa))
print(CSA_Q)

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

[[-0.65443587 -0.66627503 -0.35747905]
 [-0.23565623 -0.26950373  0.93372045]
 [-0.71845656  0.69530232  0.01936105]]
Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -44.06170589854923 chi: 88.8906243171395 xi: 69.05043204965092 



In [71]:
# Eigenvectors for CSA --> EFG Frame from ASICS
psi_A = 85.2
chi_A = 49.5
xi_A = 201.9

eigenvectors_csa_Q = Rabc(psi_A, chi_A, xi_A)
print('Eigenvectors for CSA --> EFG Frame from ASICS\n',eigenvectors_csa_Q)

get_euler_angles(eigenvectors_csa_Q)

Eigenvectors for CSA --> EFG Frame from ASICS
 [[ 0.32125695 -0.63167892  0.70553222]
 [ 0.944852    0.16374729 -0.28362213]
 [ 0.06362913  0.75773911  0.64944805]]


(85.2, 49.5, 21.900000000000006)

In [72]:
# Find Quad tensor in PAS

# Find Quadrupolar Tensor and rotate in tenon frame
# Values from ASICS
cq = 2.23
etaq = 0.823
a1 = 290.8
b1 = 69.0
g1 = 30.0

V_PAS = np.zeros((3,3))
V_PAS[0,0] = -(1 + etaq) * cq/2
V_PAS[1,1] = -(1 - etaq) * cq/2
V_PAS[2,2] = cq

#Transformation between PAS --> Tenon frame
U = Rabc(a1, b1, g1)
V_T = np.matmul(np.matmul(U, V_PAS), np.linalg.inv(U))
print('Quadrupolar tensor in PAS: \n',V_PAS)
print('Quadrupolar tensor in Tenon: \n',V_T)

Quadrupolar tensor in PAS: 
 [[-2.032645  0.        0.      ]
 [ 0.       -0.197355  0.      ]
 [ 0.        0.        2.23    ]]
Quadrupolar tensor in Tenon: 
 [[ 0.77701673 -1.70687689 -1.05475362]
 [-1.70687689 -0.68969148 -0.04781122]
 [-1.05475362 -0.04781122 -0.08732524]]


In [73]:
A = [1, 2, 3]
B = [5 * i for i in A]

print(A, B)

[1, 2, 3] [5, 10, 15]


In [74]:
# Find Rotation Matrix and Rotation angles

# *************** Calculation for CSA ************************
# Quadrupolar Tensor in PAS
Q_PAS = np.zeros((3,3))
Q_PAS[0,0] = Vxx/(2*Ispin*(2*Ispin - 1));
Q_PAS[1,1] = Vyy/(2*Ispin*(2*Ispin - 1));
Q_PAS[2,2] = Vzz/(2*Ispin*(2*Ispin - 1));

#CSA tensor in PAS
CS_PAS = np.zeros((3,3))
CS_PAS[0,0] = -csxx; 
CS_PAS[1,1] = -csyy;
CS_PAS[2,2] = -cszz;
print('Calculation for CSA Tensor: \n')

# Find eigenvalues and eigenvectors of original matrix

eigenvalues, eigenvectors = np.linalg.eig(CS_PAS) 
print('Eigenvalues of CSA (PAS) tensor \n', eigenvalues, '\n')
print('Eigenvectors of CSA (PAS) tensor \n', eigenvectors, '\n')
# Calculate the eigenvalues of the rotated matrix A_rot

eigenvalues_rot, eigenvectors_rot = np.linalg.eig(CS_T)
print('Eigenvalues of CSA (Tenon) tensor \n', eigenvalues_rot, '\n')
print('Eigenvectors of CSA (Tenon) tensor \n', eigenvectors_rot, '\n')

b = np.degrees(np.arccos(CS_T[2,2]))
a = np.degrees(np.arctan(CS_T[2,1]/CS_T[2,0]))
g = np.degrees(np.arctan(-CS_T[1,2]/CS_T[0,2]))
print("Calculated Euler angles (degrees):")
print(a, b, g,'\n')






Calculation for CSA Tensor: 

Eigenvalues of CSA (PAS) tensor 
 [ 37.3976574   12.10613992 -22.21145688] 

Eigenvectors of CSA (PAS) tensor 
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]] 

Eigenvalues of CSA (Tenon) tensor 
 [ 3.73976574e-05  1.21061399e-05 -2.22114569e-05] 

Eigenvectors of CSA (Tenon) tensor 
 [[ 0.18009761 -0.98364235 -0.00354775]
 [-0.71242587 -0.12795128 -0.68998394]
 [-0.67824349 -0.12679197  0.72381598]] 

Calculated Euler angles (degrees):
-84.16334255551028 89.99966990295337 84.16334255551028 

